# 第5章: 次元削減によるデータ圧縮

この Notebook は、原本 `machine-learning-book/ch05/ch05.ipynb` を最新の NumPy / scikit-learn / Matplotlib 環境で継続検証できる形に移行したものです。
PCA、LDA、t-SNE という章の主要トピックを保ちながら、外部データ取得や古い API に依存しない構成へ整理しています。


## この Notebook で確認すること

- 現在の `uv` 環境で Python と主要パッケージのバージョンを確認する。
- 原本サブモジュールの図版を読み取り専用で参照する。
- Wine データセットに対して PCA を手計算と scikit-learn の両方で再現する。
- LDA を散布行列ベースで確認し、scikit-learn 実装と比較する。
- Digits データセットに対して t-SNE を実行し、2 次元埋め込みを可視化する。
- `pytest --nbmake` によるヘッドレス実行で完走することを確認する。


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.datasets import load_digits, load_wine
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'scikit-learn']

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / '.github/workflows/ci.yml').exists() and (candidate / 'machine-learning-book').exists():
            return candidate
    raise FileNotFoundError('リポジトリルートを特定できませんでした')

REPO_ROOT = find_repo_root(Path.cwd())
FIG_DIR = REPO_ROOT / 'machine-learning-book' / 'ch05' / 'figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'原本 ch05 図版ディレクトリ: {FIG_DIR}')


In [ ]:
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

書籍の概念図は `machine-learning-book/ch05/figures/` から読み取り専用で参照し、実装コードは `src/` 側に移します。
次元削減の直感を与える図版を残しつつ、Notebook 本体は CI で安定して実行できるようにします。


In [ ]:
selected_figures = [
    ('05_01.png', 420),
    ('05_06.png', 420),
    ('05_11.png', 520),
    ('05_12.png', 520),
]

for name, width in selected_figures:
    figure_path = FIG_DIR / name
    print(name)
    display(Image(filename=str(figure_path), width=width))


## Wine データセットの準備

原本では UCI の CSV を読み込んでいましたが、移行版では `scikit-learn` 同梱の Wine データセットを使ってローカル完結にします。
PCA と LDA の両方で使うため、train/test 分割と標準化をここで共通化します。


In [ ]:
wine = load_wine(as_frame=True)
df_wine = wine.frame.copy()
df_wine.rename(columns={'target': 'Class label'}, inplace=True)
df_wine['Class label'] = df_wine['Class label'] + 1

X = df_wine.drop(columns=['Class label']).to_numpy()
y = df_wine['Class label'].to_numpy()
feature_names = df_wine.drop(columns=['Class label']).columns.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=0,
)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

print('クラス分布:', np.bincount(y)[1:])
print('train/test shape:', X_train.shape, X_test.shape)
df_wine.head()


In [ ]:
def plot_decision_regions(X: np.ndarray, y: np.ndarray, classifier, resolution: float = 0.02):
    markers = ('o', 's', '^', 'v', '<')
    colors = ('tab:red', 'tab:blue', 'tab:green', 'gray', 'cyan')
    cmap = ListedColormap(colors[: len(np.unique(y))])

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, resolution),
        np.arange(x2_min, x2_max, resolution),
    )
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    lab = classifier.predict(grid).reshape(xx1.shape)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.contourf(xx1, xx2, lab, alpha=0.25, cmap=cmap)
    for idx, cl in enumerate(np.unique(y)):
        ax.scatter(
            X[y == cl, 0], X[y == cl, 1],
            c=colors[idx], marker=markers[idx], edgecolor='black', alpha=0.85,
            label=f'Class {cl}'
        )
    ax.set_xlim(xx1.min(), xx1.max())
    ax.set_ylim(xx2.min(), xx2.max())
    return fig, ax


## PCA を段階的に確認する

まずは共分散行列の固有値分解から、主成分の寄与率と射影行列を手で求めます。
その後、scikit-learn の `PCA` と比較します。


In [ ]:
cov_mat = np.cov(X_train_std.T)
eigen_vals, eigen_vecs = np.linalg.eigh(cov_mat)
order = np.argsort(eigen_vals)[::-1]
eigen_vals = eigen_vals[order]
eigen_vecs = eigen_vecs[:, order]

var_exp = eigen_vals / eigen_vals.sum()
cum_var_exp = np.cumsum(var_exp)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 14), var_exp, align='center', label='Individual explained variance')
ax.step(range(1, 14), cum_var_exp, where='mid', label='Cumulative explained variance')
ax.set_xlabel('Principal component index')
ax.set_ylabel('Explained variance ratio')
ax.legend(loc='best')
plt.show()
plt.close(fig)

w_pca = np.hstack((eigen_vecs[:, [0]], eigen_vecs[:, [1]]))
X_train_pca_manual = X_train_std.dot(w_pca)

fig, ax = plt.subplots(figsize=(6, 4))
for label, color, marker in zip(np.unique(y_train), ['tab:red', 'tab:blue', 'tab:green'], ['o', 's', '^']):
    ax.scatter(X_train_pca_manual[y_train == label, 0], X_train_pca_manual[y_train == label, 1],
               c=color, marker=marker, label=f'Class {label}')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('手計算 PCA による射影')
ax.legend(loc='lower left')
plt.show()
plt.close(fig)

pd.DataFrame({'explained_variance_ratio': var_exp[:5], 'cumulative': cum_var_exp[:5]})


In [ ]:
pca_full = PCA()
X_train_pca_full = pca_full.fit_transform(X_train_std)

pca_2 = PCA(n_components=2)
X_train_pca = pca_2.fit_transform(X_train_std)
X_test_pca = pca_2.transform(X_test_std)

fig, ax = plot_decision_regions(X_train_pca, y_train, classifier=LogisticRegression(random_state=1, max_iter=200).fit(X_train_pca, y_train))
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('PCA + Logistic Regression (train)')
ax.legend(loc='lower left')
plt.show()
plt.close(fig)

lr_pca = LogisticRegression(random_state=1, max_iter=200)
lr_pca.fit(X_train_pca, y_train)
fig, ax = plot_decision_regions(X_test_pca, y_test, classifier=lr_pca)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('PCA + Logistic Regression (test)')
ax.legend(loc='lower left')
plt.show()
plt.close(fig)

loadings_manual = eigen_vecs * np.sqrt(eigen_vals)
loadings_sklearn = pca_full.components_.T * np.sqrt(pca_full.explained_variance_)

pd.DataFrame(
    {
        'feature': feature_names,
        'manual_pc1_loading': loadings_manual[:, 0],
        'sklearn_pc1_loading': loadings_sklearn[:, 0],
    }
).head()


## LDA を段階的に確認する

次に、クラス平均、within-class scatter、between-class scatter を求めて線形判別方向を導きます。
その後、scikit-learn の `LinearDiscriminantAnalysis` で同じ問題を解きます。


In [ ]:
mean_vecs = [np.mean(X_train_std[y_train == label], axis=0) for label in range(1, 4)]

d = X_train_std.shape[1]
S_W = np.zeros((d, d))
for label in range(1, 4):
    class_scatter = np.cov(X_train_std[y_train == label].T)
    S_W += class_scatter

mean_overall = np.mean(X_train_std, axis=0).reshape(d, 1)
S_B = np.zeros((d, d))
for idx, mean_vec in enumerate(mean_vecs):
    n = X_train_std[y_train == idx + 1, :].shape[0]
    mean_vec = mean_vec.reshape(d, 1)
    S_B += n * (mean_vec - mean_overall).dot((mean_vec - mean_overall).T)

eigvals_lda, eigvecs_lda = np.linalg.eig(np.linalg.pinv(S_W).dot(S_B))
order = np.argsort(np.abs(eigvals_lda))[::-1]
eigvals_lda = eigvals_lda[order].real
eigvecs_lda = eigvecs_lda[:, order].real

discr = eigvals_lda / eigvals_lda.sum()
cum_discr = np.cumsum(discr)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(discr) + 1), discr, align='center', label='Individual discriminability')
ax.step(range(1, len(cum_discr) + 1), cum_discr, where='mid', label='Cumulative discriminability')
ax.set_xlabel('Linear discriminants')
ax.set_ylabel('Discriminability ratio')
ax.set_ylim(-0.1, 1.1)
ax.legend(loc='best')
plt.show()
plt.close(fig)

w_lda = np.hstack((eigvecs_lda[:, [0]], eigvecs_lda[:, [1]]))
X_train_lda_manual = X_train_std.dot(w_lda)

fig, ax = plt.subplots(figsize=(6, 4))
for label, color, marker in zip(np.unique(y_train), ['tab:red', 'tab:blue', 'tab:green'], ['o', 's', '^']):
    ax.scatter(X_train_lda_manual[y_train == label, 0], -X_train_lda_manual[y_train == label, 1],
               c=color, marker=marker, label=f'Class {label}')
ax.set_xlabel('LD 1')
ax.set_ylabel('LD 2')
ax.set_title('手計算 LDA による射影')
ax.legend(loc='lower right')
plt.show()
plt.close(fig)


In [ ]:
lda = LDA(n_components=2)
X_train_lda = lda.fit_transform(X_train_std, y_train)
X_test_lda = lda.transform(X_test_std)

lr_lda = LogisticRegression(random_state=1, max_iter=200)
lr_lda.fit(X_train_lda, y_train)

fig, ax = plot_decision_regions(X_train_lda, y_train, classifier=lr_lda)
ax.set_xlabel('LD 1')
ax.set_ylabel('LD 2')
ax.set_title('LDA + Logistic Regression (train)')
ax.legend(loc='lower left')
plt.show()
plt.close(fig)

fig, ax = plot_decision_regions(X_test_lda, y_test, classifier=lr_lda)
ax.set_xlabel('LD 1')
ax.set_ylabel('LD 2')
ax.set_title('LDA + Logistic Regression (test)')
ax.legend(loc='lower left')
plt.show()
plt.close(fig)


## t-SNE による非線形次元削減

最後に、Digits データセットを 2 次元へ埋め込みます。`TSNE` はランダム性を持つため、`random_state` を固定し、現行 API の引数を使います。


In [ ]:
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

fig, axes = plt.subplots(1, 4, figsize=(8, 2.2))
for i in range(4):
    axes[i].imshow(digits.images[i], cmap='Greys')
    axes[i].axis('off')
plt.show()
plt.close(fig)

tsne = TSNE(n_components=2, init='pca', learning_rate='auto', random_state=123, perplexity=30)
X_digits_tsne = tsne.fit_transform(X_digits)

fig, ax = plt.subplots(figsize=(8, 8))
for i in range(10):
    ax.scatter(X_digits_tsne[y_digits == i, 0], X_digits_tsne[y_digits == i, 1], s=12, alpha=0.7)

for i in range(10):
    xtext, ytext = np.median(X_digits_tsne[y_digits == i, :], axis=0)
    txt = ax.text(xtext, ytext, str(i), fontsize=20)
    txt.set_path_effects([PathEffects.Stroke(linewidth=4, foreground='white'), PathEffects.Normal()])

ax.set_title('t-SNE projection of Digits')
plt.show()
plt.close(fig)

print('Digits shape:', X_digits.shape)


## まとめ

第5章の移行版では、原本の次元削減フローを以下の形で継続検証可能にしました。

- Wine データセットは `scikit-learn` 同梱データを使い、外部 URL やローカル CSV 依存をなくした。
- PCA と LDA は手計算の流れを残しつつ、scikit-learn 実装と比較できる形にした。
- t-SNE は Digits データセットに対して現行 API で再現し、埋め込み可視化を維持した。
- 原本の図版は読み取り専用サブモジュールから再利用し、Notebook 本体は `src/` 側に配置した。
